In [1]:
from syncode import Syncode
import textwrap
from enum import StrEnum
from pathlib import Path
import subprocess
import tempfile

c:\Users\stett\~Repos\syncode-cs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "microsoft/phi-2"

# Load the Syncode augmented model
syn_llm = Syncode(
    model=model_name,
    mode='grammar_strict',
    parse_output_only=False,
    grammar="./c_grammar/c11.lark",
)

[2026-11-20 21:28:08,520-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.88s/it]


In [8]:
def test_examples(path: Path, verbose: bool = False):
    results = {}
    for source_path in sorted(path.glob("*.c")):
        # Read the source code stub from the file and generate a completion using the Syncode model
        example_stub = source_path.read_text(encoding="utf-8")
        completion = syn_llm.infer(example_stub, stop_words=["\n}"])[0] # type: ignore
        completed_program = example_stub + completion

        # Write the completed program to a temporary file and run GCC syntax check
        with tempfile.TemporaryDirectory() as temp_dir:
            generated_path = Path(temp_dir) / source_path.name
            generated_path.write_text(completed_program, encoding="utf-8")
            result = subprocess.run(
                ["gcc", "-c", "-fsyntax-only", "-Wno-implicit-function-declaration", str(generated_path)],
                capture_output=True,
                text=True,
                check=False,
            )

        # Store the results for this source file
        if_passed = (result.returncode == 0)
        results[source_path.name] = {
            "completed_program": completed_program,
            "gcc_result": result,
            "passed": if_passed,
        }
    
        if verbose:
            print(f"--- {source_path.name} ---")
            print(completed_program)
            if result.stdout:
                print(result.stdout)
            if result.stderr:
                print(result.stderr)
            print(f"GCC syntax check: {if_passed}")
            print()

    return results


In [7]:
test_examples(Path("./c_stubs/typedef_advanced"), verbose=True)

--- typedef_buffer.c ---
typedef unsigned char Byte;
typedef Byte Octet;
typedef struct
{
    Octet data[8];
    unsigned int length;
} Buffer;

Octet buffer_checksum(const Buffer *buffer);

int main(void)
{
    Buffer buffer = {{1, 2, 3, 4, 5, 6, 7, 8}, 8};
    Octet checksum;

    checksum = buffer_checksum(&buffer);

    return 0;
}

// Return the checksum of the bytes stored in the buffer.
Octet buffer_checksum(const Buffer *buffer)
{
    unsigned int sum = 0;
    for (int i = 0; i < buffer->length; i++)
    {
        sum += buffer->data[i];
    }

    return sum % 256;
}
GCC syntax check: True

--- typedef_point.c ---
typedef double Coordinate;
typedef struct
{
    Coordinate x;
    Coordinate y;
} Point;

Coordinate point_distance(Point first, Point second);

int main(void)
{
    Point first = {0.0, 0.0};
    Point second = {3.0, 4.0};
    Coordinate distance;

    distance = point_distance(first, second);

    return 0;
}

// Return the distance between two points.
Coordinate po

{'typedef_buffer.c': {'completed_program': 'typedef unsigned char Byte;\ntypedef Byte Octet;\ntypedef struct\n{\n    Octet data[8];\n    unsigned int length;\n} Buffer;\n\nOctet buffer_checksum(const Buffer *buffer);\n\nint main(void)\n{\n    Buffer buffer = {{1, 2, 3, 4, 5, 6, 7, 8}, 8};\n    Octet checksum;\n\n    checksum = buffer_checksum(&buffer);\n\n    return 0;\n}\n\n// Return the checksum of the bytes stored in the buffer.\nOctet buffer_checksum(const Buffer *buffer)\n{\n    unsigned int sum = 0;\n    for (int i = 0; i < buffer->length; i++)\n    {\n        sum += buffer->data[i];\n    }\n\n    return sum % 256;\n}',
  'gcc_result': CompletedProcess(args=['gcc', '-c', '-fsyntax-only', '-Wno-implicit-function-declaration', 'C:\\Users\\stett\\AppData\\Local\\Temp\\tmp1dw13ugj\\typedef_buffer.c'], returncode=0, stdout='', stderr=''),
  'passed': True},
 'typedef_point.c': {'completed_program': 'typedef double Coordinate;\ntypedef struct\n{\n    Coordinate x;\n    Coordinate y;\n}